In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.compose import ColumnTransformer

In [18]:
df = pd.read_csv('../3/train.csv')[['Age','Fare','SibSp','Parch','Survived']]

In [19]:
df.dropna(inplace=True)
# dropna() → dataset me jitni bhi rows me missing values (NaN) hain, unko remove karta hai
# inplace=True → changes original DataFrame (df) me hi apply ho jate hain, new copy nahi banti

In [20]:
df.head()

,Age,Fare,SibSp,Parch,Survived
0,22.0,7.2500,1,0,0
1,38.0,71.2833,1,0,1
2,26.0,7.9250,0,0,1
3,35.0,53.1000,1,0,1
4,35.0,8.0500,0,0,0


In [21]:
df['family'] = df['SibSp'] + df['Parch']

In [22]:
df.head()

,Age,Fare,SibSp,Parch,Survived,family
0,22.0,7.2500,1,0,0,1
1,38.0,71.2833,1,0,1,1
2,26.0,7.9250,0,0,1,0
3,35.0,53.1000,1,0,1,1
4,35.0,8.0500,0,0,0,0


In [23]:
df.drop(columns=['SibSp', 'Parch'], inplace=True)

In [24]:
df.head()

,Age,Fare,Survived,family
0,22.0,7.2500,0,1
1,38.0,71.2833,1,1
2,26.0,7.9250,1,0
3,35.0,53.1000,1,1
4,35.0,8.0500,0,0


In [25]:
X = df.drop(columns=['Survived'])
y = df['Survived']

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y,
    test_size = 0.2, 
    random_state = 42
)

In [27]:
X_train.head()

,Age,Fare,family
328,31.0,20.5250,2
73,26.0,14.4542,1
253,30.0,16.1000,1
719,33.0,7.7750,0
666,25.0,13.0000,0


In [28]:
# Without binarization

clf = DecisionTreeClassifier()

clf.fit(X_train,y_train)

y_pred = clf.predict(X_test)

accuracy_score(y_test,y_pred)

0.6293706293706294

In [31]:
from sklearn.model_selection import cross_val_score

In [32]:
np.mean(cross_val_score(DecisionTreeClassifier(),X,y,cv=10,scoring='accuracy'))

np.float64(0.647085289514867)

In [ ]:
# Applying Binarization

from sklearn.preprocessing import Binarizer
# Binarizer import kiya ja raha hai sklearn library se
# iska use continuous numeric values ko binary form (0/1) me convert karne ke liye hota hai
# example: value > threshold → 1, value <= threshold → 0

In [43]:
trf = ColumnTransformer([
    ('bin', Binarizer(copy=False), ['family'])
], remainder='passthrough')
# ColumnTransformer → different columns par different preprocessing apply karta hai

# ('bin', Binarizer(copy=False), ['Family'])
# 'bin' → transformer ka naam
# Binarizer(copy=False) → Family column ko 0/1 me convert karega
# copy=False → data copy nahi banata, original reference use karta hai (memory efficient)
# ['Family'] → sirf Family column par binarization apply hoga

# remainder='passthrough'
# jo columns mention nahi kiye gaye (baaki columns) unko same as-it-is pass kar diya jayega

In [44]:
X_train_trf = trf.fit_transform(X_train)
X_test_trf = trf.transform(X_test)

In [45]:
pd.DataFrame(X_train_trf,columns=['family','Age','Fare'])

,family,Age,Fare
0,1.0,31.0,20.5250
1,1.0,26.0,14.4542
2,1.0,30.0,16.1000
3,0.0,33.0,7.7750
4,0.0,25.0,13.0000
...,...,...,...
566,1.0,46.0,61.1750
567,0.0,25.0,13.0000
568,0.0,41.0,134.5000
569,1.0,33.0,20.5250


In [46]:
clf = DecisionTreeClassifier()
clf.fit(X_train_trf, y_train)
y_pred2 = clf.predict(X_test_trf)

accuracy_score(y_test, y_pred2)

0.6153846153846154

In [47]:
X_trf = trf.fit_transform(X)
np.mean(cross_val_score(DecisionTreeClassifier(), X_trf, y, cv=10, scoring='accuracy'))

np.float64(0.6318075117370892)